# Estimated time of execution: 30 minutes

In [19]:
import time
import sqlite3
from pathlib import Path
import pandas as pd
import yfinance as yf


In [20]:
TICKERS_CSV = Path('/home/batenkh/diploma/dataset/tickers.csv')
DB_PATH = Path('/home/batenkh/diploma/dataset/stock_prices_20y.db')
SQL_DUMP_PATH = Path('/home/batenkh/diploma/dataset/stock_prices_20y.sql')

# Download settings
BATCH_PAUSE_SEC = 1.0  
MAX_RETRIES = 3
BACKOFF_SEC = 2.0


end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=20)

start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')

print('Date range:', start_str, 'to', end_str)


Date range: 2006-02-22 to 2026-02-22


In [21]:
# Load tickers
if not TICKERS_CSV.exists():
    raise FileNotFoundError(f'Missing tickers file: {TICKERS_CSV}')

tickers_df = pd.read_csv(TICKERS_CSV)
if 'ticker' in tickers_df.columns:
    tickers = tickers_df['ticker'].astype(str).str.strip().tolist()
elif tickers_df.shape[1] == 1:
    tickers = tickers_df.iloc[:, 0].astype(str).str.strip().tolist()
else:
    # Try common column names
    for col in ['symbol', 'Ticker', 'Symbol']:
        if col in tickers_df.columns:
            tickers = tickers_df[col].astype(str).str.strip().tolist()
            break
    else:
        raise ValueError('Ticker column not found. Use a single-column CSV or name the column ticker/symbol.')

tickers = [t for t in tickers if t]
print('Total tickers:', len(tickers))
if len(tickers) < 1000:
    raise ValueError('Need at least 1000 tickers in the CSV.')



Total tickers: 1000


In [22]:
# Initialize database
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

conn.execute(
    '''
    CREATE TABLE IF NOT EXISTS prices (
        ticker TEXT NOT NULL,
        date TEXT NOT NULL,
        open REAL,
        high REAL,
        low REAL,
        close REAL,
        adj_close REAL,
        volume INTEGER
    )
    '''
)

conn.execute(
    'CREATE INDEX IF NOT EXISTS idx_prices_ticker_date ON prices(ticker, date)'
)
conn.commit()

def already_loaded_tickers(connection):
    rows = connection.execute('SELECT DISTINCT ticker FROM prices').fetchall()
    return set(r[0] for r in rows)

loaded = already_loaded_tickers(conn)
print('Tickers already in DB:', len(loaded))


Tickers already in DB: 0


In [23]:
def fetch_history(ticker):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = yf.download(
                ticker,
                start=start_str,
                end=end_str,
                progress=False,
                auto_adjust=False,
                threads=False,
            )
            if df.empty:
                return pd.DataFrame()

            # yfinance can return MultiIndex columns; normalize to single level
            if isinstance(df.columns, pd.MultiIndex):
                if ticker in df.columns.get_level_values(-1):
                    df = df.xs(ticker, axis=1, level=-1)
                else:
                    df.columns = df.columns.get_level_values(0)

            df = df.reset_index()
            df['ticker'] = ticker

            # Normalize column names
            rename_map = {
                'Date': 'date',
                'Open': 'open',
                'High': 'high',
                'Low': 'low',
                'Close': 'close',
                'Adj Close': 'adj_close',
                'Adj_Close': 'adj_close',
                'Volume': 'volume',
            }
            df = df.rename(columns=rename_map)

            for col in ['open','high','low','close','adj_close','volume']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            cols = ['ticker','date','open','high','low','close','adj_close','volume']
            return df[cols]
        except Exception as exc:
            if attempt == MAX_RETRIES:
                print(f'Failed: {ticker} after {MAX_RETRIES} attempts. Error: {exc}')
                return pd.DataFrame()
            sleep_s = BACKOFF_SEC * attempt
            print(f'Retry {attempt}/{MAX_RETRIES} for {ticker} in {sleep_s:.1f}s...')
            time.sleep(sleep_s)

def save_to_db(connection, df):
    if df.empty:
        return 0
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
    df.to_sql('prices', connection, if_exists='append', index=False)
    connection.commit()
    return len(df)



In [24]:
# Main download loop
processed = 0
inserted = 0

for ticker in tickers:
    if ticker in loaded:
        continue
    df = fetch_history(ticker)
    n = save_to_db(conn, df)
    processed += 1
    inserted += n
    if processed % 10 == 0:
        print(f'Processed {processed} tickers, inserted {inserted} rows')
    time.sleep(BATCH_PAUSE_SEC)

print('Done. Total inserted rows:', inserted)


Processed 10 tickers, inserted 38302 rows
Processed 20 tickers, inserted 76850 rows
Processed 30 tickers, inserted 123594 rows
Processed 40 tickers, inserted 165968 rows
Processed 50 tickers, inserted 208157 rows
Processed 60 tickers, inserted 245868 rows
Processed 70 tickers, inserted 286654 rows
Processed 80 tickers, inserted 323316 rows
Processed 90 tickers, inserted 353822 rows
Processed 100 tickers, inserted 383777 rows
Processed 110 tickers, inserted 425646 rows
Processed 120 tickers, inserted 458531 rows
Processed 130 tickers, inserted 495000 rows
Processed 140 tickers, inserted 528790 rows
Processed 150 tickers, inserted 575959 rows
Processed 160 tickers, inserted 619714 rows
Processed 170 tickers, inserted 655871 rows
Processed 180 tickers, inserted 699856 rows
Processed 190 tickers, inserted 742800 rows
Processed 200 tickers, inserted 778996 rows
Processed 210 tickers, inserted 823915 rows
Processed 220 tickers, inserted 863560 rows
Processed 230 tickers, inserted 902844 rows

In [25]:
# Export SQL dump (can be large)
with open(SQL_DUMP_PATH, 'w', encoding='utf-8') as f:
    for line in conn.iterdump():
        f.write(f'{line}\n')

print('SQL dump written to', SQL_DUMP_PATH)


SQL dump written to /home/batenkh/diploma/dataset/stock_prices_20y.sql
